In [2]:
import ee
import geemap

# ── Authenticate & initialise the Earth Engine API ───────────────────────────
# This must be called before any ee.* objects are constructed.
# On first run it opens a browser auth flow; subsequent runs use cached creds.

ee.Authenticate()
ee.Initialize()

In [3]:
# A blank map template should appear. If not, then the packages were not correctly instatlled

Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [14]:
# NEW DEM 



# =============================================================================
# 1m DEM EXTRACTION — CLEAN AOI-BASED EXPORT (PARTITION READY)
# =============================================================================

import ee
import geemap

ee.Initialize()
# ^ Starts Earth Engine session so all dataset and export calls work.

# =============================================================================
# USER CONFIGURATION
# =============================================================================

ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ AOI polygon FeatureCollection defining the corridor / study boundary.

Export_CRS = 'EPSG:5070'
# ^ Output projection (Albers Equal Area Conic, meters-based CRS for CONUS).

FOLDER = 'GEE_Exports_DEM'
# ^ Google Drive folder for exported GeoTIFF tiles.

FILE_PREFIX = 'MN_I-90_DEM'
# ^ Base filename for exported DEM tiles.

EXPORT_SCALE = 1
# ^ 1 meter DEM resolution (native 3DEP resolution preserved).

NODATA_VALUE = -9999
# ^ Standard GIS NoData value for ArcGIS / QGIS compatibility.

# =============================================================================
# LOAD DATA
# =============================================================================

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads AOI polygon(s) from Earth Engine assets.

dissolved = export_poly.union().geometry()
# ^ Dissolves all AOI features into a single geometry for consistent filtering.

dem_mosaic = (
    ee.ImageCollection('USGS/3DEP/1m')
    .filterBounds(export_poly)
    # ^ Keeps only DEM tiles intersecting AOI to reduce processing load.

    .mosaic()
    # ^ Stitches overlapping 3DEP tiles into a seamless raster.

    .select(0)
    # ^ Selects elevation band (first band in dataset).
)

# =============================================================================
# PREPARE DEM
# =============================================================================

dem_export = (
    dem_mosaic
    .clip(export_poly)
    # ^ Restricts raster strictly to AOI boundary (removes external pixels).

    .reproject(crs=Export_CRS, scale=EXPORT_SCALE)
    # ^ Forces consistent grid alignment and resolution in output CRS.

    .rename('elevation_m')
    # ^ Assigns meaningful band name for GIS interpretation.
)

ready_to_export = (
    dem_export
    .toFloat()
    # ^ Ensures 32-bit float output for continuous elevation values.

    .unmask(NODATA_VALUE)
    # ^ Fills masked pixels with GIS-friendly NoData value.
)

# =============================================================================
# INTERACTIVE MAP (QUALITY CONTROL)
# =============================================================================

Map = geemap.Map()
Map.centerObject(export_poly, 11)
# ^ Centers map view on AOI at regional scale.

Map.add_basemap('SATELLITE')
# ^ Adds high-resolution imagery for visual verification.

Map.addLayer(
    dem_mosaic.clip(export_poly),
    {
        'min': 0,
        'max': 500,
        # ^ Visualization range for terrain interpretation.

        'palette': [
            '006633',  # low elevation (valleys)
            'E5FFCC',  # low-mid
            'FFCC00',  # mid elevation
            '662A00',  # high terrain
            'FFFFFF'   # highest peaks / bright highlight
        ]
    },
    'DEM'
)

Map.addLayer(export_poly, {'color': 'FF1744'}, 'AOI')
# ^ Displays AOI boundary overlay.

display(Map)
# ^ Renders interactive map in notebook environment.

# =============================================================================
# PARTITIONING LOGIC (AOI-BASED SCALING)
# =============================================================================

projected_bbox = dissolved.bounds(1, Export_CRS)
# ^ Computes bounding box of AOI in projected CRS (meters).

coords = projected_bbox.coordinates().get(0).getInfo()
# ^ Converts Earth Engine geometry to Python-readable coordinates.

xmin = coords[0][0]
ymin = coords[0][1]
xmax = coords[2][0]
ymax = coords[2][1]

bbox_area_m2 = projected_bbox.area(1).getInfo()
# ^ Total bounding box area in square meters.

BYTES_PER_PX = 4
BYTES_PER_GB = 1e9
# ^ Used for raster size estimation (Float32 = 4 bytes per pixel).

total_gb = (bbox_area_m2 / (EXPORT_SCALE ** 2)) * BYTES_PER_PX / BYTES_PER_GB
# ^ Estimates total DEM size in GB based on pixel density.

TARGET_GB = 10.0
# ^ Maximum recommended size per export task.

n_tasks = max(1, int(total_gb / TARGET_GB) + 1)
# ^ Automatically determines number of export partitions needed.

step_x = (xmax - xmin) / n_tasks
# ^ Width of each partition in projected coordinate space.

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"Estimated size (GB): {total_gb:.2f}")
print(f"Number of tasks:     {n_tasks}")
print(f"CRS:                 {Export_CRS}")
print(f"Resolution:          {EXPORT_SCALE} m")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# =============================================================================
# EXPORT TASKS (PARTITIONED AOI STRIPS)
# =============================================================================

for i in range(n_tasks):

    x0 = xmin + step_x * i
    x1 = xmin + step_x * (i + 1)
    # ^ Defines spatial extent of each vertical partition.

    strip_geom = (
        ee.Geometry.Rectangle([x0, ymin, x1, ymax], Export_CRS, False)
        .intersection(dissolved, ee.ErrorMargin(1))
    )
    # ^ Ensures export region is clipped to actual AOI (not full rectangle).

    task = ee.batch.Export.image.toDrive(
        image=ready_to_export,
        # ^ Final processed DEM image.

        description=f"{FILE_PREFIX}_{i:03d}",
        # ^ Task name in Earth Engine Task Manager.

        folder=FOLDER,
        # ^ Google Drive output folder.

        fileNamePrefix=f"{FILE_PREFIX}_{i:03d}",
        # ^ Output filename base.

        region=strip_geom,
        # ^ Spatial extent of export task.

        scale=EXPORT_SCALE,
        # ^ Output resolution (meters).

        crs=Export_CRS,
        # ^ Output coordinate system.

        maxPixels=int(1e13)
        # ^ Prevents export failure due to pixel limits.
    )

    task.start()
    # ^ Submits task to Earth Engine queue (runs asynchronously).

    print(f"[{i+1}/{n_tasks}] {FILE_PREFIX}_{i:03d}")

# =============================================================================
# FINAL STATUS
# =============================================================================

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("✓ DEM EXPORT TASKS SUBMITTED SUCCESSFULLY")
print("Monitor at: https://code.earthengine.google.com/tasks")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

Map(center=[43.6353558671585, -95.70741888179828], controls=(WidgetControl(options=['position', 'transparent_b…

DEM export started


In [ ]:
## OLD 

# 1m DEM Extraction


# ═════════════════════════════════════════════════════════════════════════════
# 🎯 USER CONFIGURATION  —  only edit values in this block
# ═════════════════════════════════════════════════════════════════════════════

ASSET_ID    = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ Your GEE asset path. Must be a FeatureCollection (polygon(s)).
#   Format: 'projects/<your-project>/assets/<asset-name>'

Export_CRS  = 'EPSG:5070'
# ^ Coordinate Reference System for the output raster.
#   26910 = NAD83 / UTM zone 10N (metres). Change to match your study area.
#   Rule of thumb: pick a projected (metre-based) CRS, not geographic (degrees).

FOLDER      = 'GEE_Exports_DEM'
# ^ Google Drive folder where finished GeoTIFFs will land.
#   GEE creates this folder automatically if it doesn't exist.

FILE_PREFIX = 'MN_I-90'
# ^ Every output file will be named  <FILE_PREFIX>_000.tif, _001.tif, etc.

# ═════════════════════════════════════════════════════════════════════════════
# CONSTANTS  —  no need to change these
# ═════════════════════════════════════════════════════════════════════════════

BYTES_PER_PX = 4
# ^ Float32 encoding = 4 bytes per pixel. All GEE exports default to Float32
#   unless you call .toByte() / .toInt16() etc. beforehand.

BYTES_PER_GB = 1e9
# ^ 1 GB = 1 000 000 000 bytes (SI definition; GEE uses this convention).

EXPORT_SCALE = 1
# ^ Output pixel size in metres. 1 = native 3DEP 1 m resolution.
#   Increase (e.g. 3 or 10) to drastically reduce file size at lower detail.

TARGET_GB = 10.0
# ^ Maximum estimated size per export task. 10 GB gives comfortable headroom
#   below GEE's ~20 GB limit. Lower this if individual tasks are still failing.

# ═════════════════════════════════════════════════════════════════════════════
# LOAD INPUT DATA
# ═════════════════════════════════════════════════════════════════════════════

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads your polygon asset from GEE. Each Feature in this collection
#   defines the AOI boundary used for clipping and export.

dissolved = export_poly.union().geometry()
# ^ Merges all features into a single geometry. Used as the spatial filter
#   when loading the DEM — ensures we only pull tiles that overlap our AOI,
#   not the entire 3DEP catalogue. Also used as the sharding boundary below.

dem_mosaic = (
    ee.ImageCollection('USGS/3DEP/1m')   # 1-metre lidar-derived elevation tiles
      .filterBounds(export_poly)          # keep only tiles touching our AOI
      .mosaic()                           # stitch tiles into one seamless Image
      .select(0)                          # band 0 = elevation in metres
)
# ^ 3DEP (3D Elevation Program) is USGS's national elevation dataset.
#   The collection is organised as many small tiles; .mosaic() blends them
#   into a single raster, prioritising tiles listed first in the collection.

# ═════════════════════════════════════════════════════════════════════════════
# PREPARE DEM FOR EXPORT
# ═════════════════════════════════════════════════════════════════════════════

dem_export = (
    dem_mosaic
      .reproject(crs=Export_CRS, scale=EXPORT_SCALE)
      # ^ Reproject to the target CRS and pixel size before export.
      #   Without this, GEE may compute at a coarser internal resolution
      #   then resample — producing a blurry result at export time.
      .rename('elevation_m')
      # ^ Names the output band. 'elevation_m' makes units explicit
      #   when the file is opened in ArcGIS / QGIS / Python.
)

# ── Masking: mark pixels outside the ROW as -1 (NoData sentinel) ─────────────
ready_to_export = dem_export.clip(export_poly).unmask(-1).toFloat()
# ^ Pipeline:
#   .clip(export_poly)  — pixels outside the polygon become masked (null)
#   .unmask(-1)         — fill ALL masked pixels with -1, which covers:
#                           • areas outside the polygon boundary
#                           • any internal gaps where 3DEP has no coverage
#   .toFloat()          — enforce Float32 band type, matching BYTES_PER_PX=4
#
# Result:
#   valid elevation pixel  →  elevation in metres (e.g. 45.3)
#   outside AOI            →  -1.0
#   no DEM coverage        →  -1.0

# ═════════════════════════════════════════════════════════════════════════════
# INTERACTIVE MAP PREVIEW
# ═════════════════════════════════════════════════════════════════════════════

Map = geemap.Map()
Map.centerObject(export_poly, 11)          # zoom level 11 ≈ city/county scale
Map.add_basemap('SATELLITE')               # Google satellite imagery as base

Map.addLayer(
    dem_mosaic.clip(export_poly),
    {'min': None, 'max': None,             # auto-stretch to data range
     'palette': ['006633', 'E5FFCC', 'FFCC00', '662A00', 'F5F5F5']},
    # palette runs low → high elevation:  dark green → pale green → yellow → brown → white
    'DEM'
)

Map.addLayer(export_poly, {'color': 'FF1744'}, 'AOI')   # red outline of your polygons

display(Map)   # renders inline in Jupyter / Colab

# ═════════════════════════════════════════════════════════════════════════════
# AUTO-SHARDED EXPORT  —  splits the AOI into N parallel tasks
# ═════════════════════════════════════════════════════════════════════════════
# Why sharding? The AOI can be hundreds of GB at 1 m resolution.
# GEE silently fails exports above ~15-20 GB, so we slice the bounding box
# into vertical strips and export each strip as its own independent task.
# All tasks run in parallel on GEE's servers — wall-clock time ≈ slowest strip.
# GEE may further split each task into multiple files (the -0000032768- suffixes
# in Drive) — this is normal and the tiles mosaic seamlessly in ArcGIS / QGIS.

# ── Compute bounding box of the full AOI in the export CRS ───────────────────
projected_bbox = dissolved.bounds(1, Export_CRS)
# ^ Axis-aligned bounding box of the dissolved AOI, computed in Export_CRS metres.
#   The '1' is the error margin in metres — fine for bounding box purposes.

coords = projected_bbox.coordinates().get(0).getInfo()
# ^ Fetches the 5 corner coordinates of the bbox rectangle to Python.
#   GEE returns them as [[xmin,ymin],[xmax,ymin],[xmax,ymax],[xmin,ymax],[xmin,ymin]]

xmin = coords[0][0]
ymin = coords[0][1]
xmax = coords[2][0]
ymax = coords[2][1]

bbox_area_m2 = projected_bbox.area(1).getInfo()
# ^ Total bounding box area in m² — used to calculate how many strips we need.

total_gb = (bbox_area_m2 / (EXPORT_SCALE ** 2) * BYTES_PER_PX) / BYTES_PER_GB
n_tasks  = max(1, int(total_gb / TARGET_GB) + 1)
# ^ Number of vertical strips needed so each strip is ≤ TARGET_GB.
#   +1 ensures we never under-count due to integer truncation.
#   max(1, ...) ensures at least one task even if the area is very small.

step_x = (xmax - xmin) / n_tasks
# ^ Width of each vertical strip in metres.

# ── Console summary ───────────────────────────────────────────────────────────
print(f'Total estimated size: {total_gb:.1f} GB')
print(f'Target size per task: {TARGET_GB} GB')
print(f'Tasks to submit:      {n_tasks}')
print(f'CRS:                  {Export_CRS}')
print(f'Export scale:         {EXPORT_SCALE}m')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

# ── Loop: one task per strip ──────────────────────────────────────────────────
for i in range(n_tasks):

    x0 = xmin + step_x * i
    x1 = xmin + step_x * (i + 1)
    # ^ Left and right edges of this strip in Export_CRS metres.

    strip_geom = (
        ee.Geometry.Rectangle([x0, ymin, x1, ymax], Export_CRS, False)
          .intersection(dissolved, ee.ErrorMargin(1))
    )
    # ^ Build the strip rectangle then intersect with the actual AOI polygon.
    #   This ensures pixels outside the ROW boundary are excluded from the
    #   export region entirely — keeps file sizes tighter than using the
    #   full rectangle as the region.
    #   ErrorMargin(1) = 1 metre tolerance for the intersection operation.

    strip_area_m2 = (x1 - x0) * (ymax - ymin)
    strip_est_gb  = (strip_area_m2 / (EXPORT_SCALE ** 2) * BYTES_PER_PX) / BYTES_PER_GB
    # ^ Per-strip size estimate using the rectangle area (conservative upper
    #   bound — actual file will be smaller after AOI intersection trims it).

    task = ee.batch.Export.image.toDrive(
        image          = ready_to_export.reproject(crs=Export_CRS, scale=EXPORT_SCALE),
        # ^ Final reproject ensures output pixels are exactly EXPORT_SCALE metres
        #   in Export_CRS — no ambiguity about GEE's internal resampling pyramid.

        description    = f'{FILE_PREFIX}_{i:03d}',
        # ^ Label shown in the GEE Tasks panel (earthengine.google.com).
        #   Zero-padded to 3 digits so tasks sort correctly up to 999 strips.

        folder         = FOLDER,
        # ^ Target Google Drive folder. GEE creates it if it doesn't exist.

        fileNamePrefix = f'{FILE_PREFIX}_{i:03d}',
        # ^ Filename (without .tif extension). GEE appends the extension.
        #   If a file with this name already exists in Drive, GEE appends a
        #   numeric suffix rather than overwriting — watch for duplicates on reruns.

        region         = strip_geom,
        # ^ Spatial extent of this strip, already intersected with the AOI.

        scale          = EXPORT_SCALE,
        # ^ Output pixel size in metres. Must match the reproject() call above.

        crs            = Export_CRS,
        # ^ Output CRS. GEE reprojects on-the-fly if source data differs.

        maxPixels      = int(1e13)
        # ^ Safety cap on total pixel count. Default is 1e8, far too low for
        #   1 m exports. 1e13 ≈ 10 trillion pixels — effectively unlimited.
    )

    task.start()
    # ^ Non-blocking — all tasks queue simultaneously and run in parallel.
    #   Each task may produce multiple files in Drive (GEE's internal tiling)
    #   but they will mosaic seamlessly in ArcGIS Pro or QGIS.

    pad = len(str(n_tasks))
    print(f'[{i+1:>{pad}}/{n_tasks}]  {FILE_PREFIX}_{i:03d}  |  ~{strip_est_gb:.1f} GB  |  x: {x0:.0f} → {x1:.0f}')

# ── Final summary ─────────────────────────────────────────────────────────────
print(f'\n✓ All {n_tasks} tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/tasks')
print('Or run:     ee.batch.Task.list()')

In [15]:
# NEW Slope 




# =============================================================================
# 1m SLOPE EXTRACTION (PERCENT) — CLEAN AOI-BASED EXPORT
# =============================================================================

import ee
import geemap

ee.Initialize()
# ^ Initializes Earth Engine session for all dataset and export operations.

# =============================================================================
# USER CONFIGURATION
# =============================================================================

ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ AOI polygon FeatureCollection defining study corridor.

Export_CRS = 'EPSG:5070'
# ^ Output projection (Albers Equal Area Conic, meters-based CRS).

FOLDER = 'GEE_Exports_Slope'
# ^ Google Drive export folder.

FILE_PREFIX = 'MN_I-90_SLOPE_PCT'
# ^ Base filename for slope outputs (percent slope).

EXPORT_SCALE = 1
# ^ Output resolution in meters (matches DEM resolution).

NODATA_VALUE = -9999
# ^ GIS-friendly NoData value.

# =============================================================================
# LOAD DATA
# =============================================================================

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads AOI polygon(s) from Earth Engine assets.

dissolved = export_poly.union().geometry()
# ^ Dissolves AOI into a single geometry for consistent processing.

dem_mosaic = (
    ee.ImageCollection('USGS/3DEP/1m')
    .filterBounds(export_poly)
    .mosaic()
    .select(0)
)
# ^ Builds seamless 1m DEM from USGS 3DEP dataset.

# =============================================================================
# SLOPE (PERCENT)
# =============================================================================

slope_deg = ee.Terrain.slope(
    dem_mosaic.reproject(crs=Export_CRS, scale=EXPORT_SCALE)
)
# ^ Computes slope in degrees from DEM using local 3x3 neighborhood gradient.
#   Output is angular slope (0–90 degrees).

slope_pct = (
    slope_deg
    .multiply(3.141592653589793 / 180)
    # ^ Convert degrees → radians

    .tan()
    # ^ tan(theta) = rise/run (dimensionless slope ratio)

    .multiply(100)
    # ^ Convert ratio → percent slope

    .rename('slope_pct')
)
# ^ Final output is percent slope (0–100+ depending on terrain steepness).

ready_to_export = (
    slope_pct
    .clip(export_poly)
    # ^ Restricts raster to AOI boundary.

    .toFloat()
    # ^ Ensures 32-bit float output.

    .unmask(NODATA_VALUE)
    # ^ Fills masked pixels with GIS NoData value.
)

# =============================================================================
# INTERACTIVE MAP (QUALITY CONTROL)
# =============================================================================

Map = geemap.Map()
Map.centerObject(export_poly, 11)
# ^ Centers map on AOI at regional scale.

Map.add_basemap('SATELLITE')
# ^ Adds imagery basemap for terrain interpretation.

Map.addLayer(
    slope_pct.clip(export_poly),
    {
        'min': 0,
        'max': 100,
        # ^ Visualization range (0% flat → steep terrain >100%)

        'palette': [
            '006633',  # flat / low slope
            'E5FFCC',  # gentle slopes
            'FFCC00',  # moderate slopes
            '662A00',  # steep slopes
            'FFFFFF'   # extreme slope highlights
        ]
    },
    'Slope (%)'
)
# ^ Visual slope layer for QA/QC.

Map.addLayer(export_poly, {'color': 'FF1744'}, 'AOI')
# ^ AOI boundary overlay.

display(Map)
# ^ Renders interactive map in notebook environment.

# =============================================================================
# EXPORT (AOI-BASED — NO BOUNDING BOX ARTIFACTS)
# =============================================================================

task = ee.batch.Export.image.toDrive(
    image=ready_to_export,
    # ^ Final processed slope raster (percent slope).

    description=FILE_PREFIX,
    # ^ Task name in Earth Engine Task Manager.

    folder=FOLDER,
    # ^ Google Drive output folder.

    fileNamePrefix=FILE_PREFIX,
    # ^ Output filename base.

    region=dissolved,
    # ^ Uses true AOI geometry (NOT bounding box) for clean export.

    scale=EXPORT_SCALE,
    # ^ Output resolution (1 m).

    crs=Export_CRS,
    # ^ Output coordinate reference system.

    maxPixels=int(1e13)
    # ^ Prevents export failure for large rasters.
)

task.start()
# ^ Submits export task asynchronously.

# =============================================================================
# FINAL STATUS
# =============================================================================

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("✓ SLOPE EXPORT STARTED")
print("AOI-based export (no bounding box)")
print("CRS:", Export_CRS)
print("Scale:", EXPORT_SCALE, "m")
print("Output: Percent slope")
print("Monitor: https://code.earthengine.google.com/tasks")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

Map(center=[43.6353558671575, -95.7074188817978], controls=(WidgetControl(options=['position', 'transparent_bg…

Slope export started


In [ ]:
# OLD

# 1m Slope Extraction 


# ═════════════════════════════════════════════════════════════════════════════
# 🎯 USER CONFIGURATION  —  only edit values in this block
# ═════════════════════════════════════════════════════════════════════════════

ASSET_ID    = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ Your GEE asset path. Must be a FeatureCollection (polygon(s)).
#   Format: 'projects/<your-project>/assets/<asset-name>'

Export_CRS  = 'EPSG:26915'
# ^ Coordinate Reference System for the output raster.
#   Change to match your study area.
#   Rule of thumb: pick a projected (metre-based) CRS, not geographic (degrees).

FOLDER      = 'I90_Slope_GEE_Exports'
# ^ Google Drive folder where finished GeoTIFFs will land.
#   GEE creates this folder automatically if it doesn't exist.

FILE_PREFIX = 'I90_Slope'
# ^ Every output file will be named  <FILE_PREFIX>_000.tif, _001.tif, etc.

# ═════════════════════════════════════════════════════════════════════════════
# CONSTANTS  —  no need to change these
# ═════════════════════════════════════════════════════════════════════════════

BYTES_PER_PX = 4
# ^ Float32 encoding = 4 bytes per pixel. All GEE exports default to Float32
#   unless you call .toByte() / .toInt16() etc. beforehand.

BYTES_PER_GB = 1e9
# ^ 1 GB = 1 000 000 000 bytes (SI definition; GEE uses this convention).

EXPORT_SCALE = 1
# ^ Output pixel size in metres. 1 = native 3DEP 1 m resolution.
#   Increase (e.g. 3 or 10) to drastically reduce file size at lower detail.

TARGET_GB = 10.0
# ^ Maximum estimated size per export task. 10 GB gives comfortable headroom
#   below GEE's ~20 GB limit. Lower this if individual tasks are still failing.

# ═════════════════════════════════════════════════════════════════════════════
# LOAD INPUT DATA
# ═════════════════════════════════════════════════════════════════════════════

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads your polygon asset from GEE. Each Feature in this collection
#   will become one separate export task later in the script.

dissolved = export_poly.union().geometry()
# ^ Merges all features into a single geometry. Used as the spatial filter
#   when loading the DEM — ensures we only pull tiles that overlap our AOI,
#   not the entire 3DEP catalogue. Also used as the sharding boundary below.

dem_mosaic = (
    ee.ImageCollection('USGS/3DEP/1m')   # 1-metre lidar-derived elevation tiles
      .filterBounds(export_poly)          # keep only tiles touching our AOI
      .mosaic()                           # stitch tiles into one seamless Image
      .select(0)                          # band 0 = elevation in metres
)
# ^ 3DEP (3D Elevation Program) is USGS's national elevation dataset.
#   The collection is organised as many small tiles; .mosaic() blends them
#   into a single raster, prioritising tiles listed first in the collection.

# ═════════════════════════════════════════════════════════════════════════════
# COMPUTE SLOPE
# ═════════════════════════════════════════════════════════════════════════════

# ── Preview layer (coarser resolution, just for map display) ─────────────────
slope_vis = (
    ee.Terrain.slope(
        dem_mosaic.reproject(crs=Export_CRS, scale=30)
        # Reproject to 30 m before slope calculation.
        # ee.Terrain.slope() uses immediate pixel neighbours to estimate rise/run,
        # so the CRS and scale here define the "neighbourhood" size.
        # 30 m is plenty for visual inspection and renders much faster on the map.
    )
    .multiply(3.14159265 / 180)   # convert degrees → radians  (slope = arctan output)
    .tan()                        # tan(radians) = rise / run  (dimensionless ratio)
    .multiply(100)                # × 100 = percent slope  (e.g. 0.45 → 45%)
    .clip(export_poly)            # trim to AOI boundary so the map looks clean
)

# ── Export layer (full 1 m resolution, written to Drive) ─────────────────────
slope_export = (
    ee.Terrain.slope(
        dem_mosaic.reproject(crs=Export_CRS, scale=EXPORT_SCALE)
        # Reproject to 1 m (or EXPORT_SCALE) so slope kernels operate at full res.
        # Without this reproject, GEE may compute slope at a coarser internal
        # resolution then resample — producing a blurry result at export time.
    )
    .multiply(3.14159265 / 180)
    .tan()
    .multiply(100)
    .rename('slope_pct')          # names the output band; shows in GIS software
)   # The 4 lines above convert from Slope from Degrees to Percentages

# ── Masking: mark pixels outside the ROW as -1 (NoData sentinel) ─────────────
ready_to_export = slope_export.clip(export_poly).unmask(-1).toFloat()
# ^ Pipeline:
#   .clip(export_poly)  — pixels outside the polygon become masked (null)
#   .unmask(-1)         — fill ALL masked pixels with -1, which covers:
#                           • areas outside the polygon boundary
#                           • any internal gaps where 3DEP has no coverage
#   .toFloat()          — enforce Float32 band type, matching BYTES_PER_PX=4
#
# Result:
#   valid slope pixel  →  0.0 to ~100.0  (percent slope)
#   outside AOI        →  -1.0
#   no DEM coverage    →  -1.0

# ═════════════════════════════════════════════════════════════════════════════
# INTERACTIVE MAP PREVIEW
# ═════════════════════════════════════════════════════════════════════════════

Map = geemap.Map()
Map.centerObject(export_poly, 11)          # zoom level 11 ≈ city/county scale
Map.add_basemap('SATELLITE')               # Google satellite imagery as base

Map.addLayer(
    dem_mosaic.clip(export_poly),
    {'min': None, 'max': None,             # auto-stretch to data range
     'palette': ['006633', 'E5FFCC', 'FFCC00', '662A00', 'F5F5F5']},
    # palette runs low → high elevation:  dark green → pale green → yellow → brown → white
    'DEM'
)

Map.addLayer(
    slope_vis,
    {'min': 0, 'max': 100,                 # 0% = flat, 100% = 45° incline
     'palette': ['006633', 'E5FFCC', 'FFCC00', '662A00', 'F5F5F5']},
    'Slope (%)'
)

Map.addLayer(export_poly, {'color': 'FF1744'}, 'AOI')   # red outline of your polygons

display(Map)   # renders inline in Jupyter / Colab

# ═════════════════════════════════════════════════════════════════════════════
# AUTO-SHARDED EXPORT  —  splits the AOI into N parallel tasks
# ═════════════════════════════════════════════════════════════════════════════
# Why sharding? The AOI can be hundreds of GB at 1 m resolution.
# GEE silently fails exports above ~15-20 GB, so we slice the bounding box
# into vertical strips and export each strip as its own independent task.
# All tasks run in parallel on GEE's servers — wall-clock time ≈ slowest strip.
# GEE may further split each task into multiple files (the -0000032768- suffixes
# in Drive) — this is normal and the tiles mosaic seamlessly in ArcGIS / QGIS.

# ── Compute bounding box of the full AOI in the export CRS ───────────────────
projected_bbox = dissolved.bounds(1, Export_CRS)
# ^ Axis-aligned bounding box of the dissolved AOI, computed in Export_CRS metres.
#   The '1' is the error margin in metres — fine for bounding box purposes.

coords = projected_bbox.coordinates().get(0).getInfo()
# ^ Fetches the 5 corner coordinates of the bbox rectangle to Python.
#   GEE returns them as [[xmin,ymin],[xmax,ymin],[xmax,ymax],[xmin,ymax],[xmin,ymin]]

xmin = coords[0][0]
ymin = coords[0][1]
xmax = coords[2][0]
ymax = coords[2][1]

bbox_area_m2 = projected_bbox.area(1).getInfo()
# ^ Total bounding box area in m² — used to calculate how many strips we need.

total_gb = (bbox_area_m2 / (EXPORT_SCALE ** 2) * BYTES_PER_PX) / BYTES_PER_GB
n_tasks  = max(1, int(total_gb / TARGET_GB) + 1)
# ^ Number of vertical strips needed so each strip is ≤ TARGET_GB.
#   +1 ensures we never under-count due to integer truncation.
#   max(1, ...) ensures at least one task even if the area is very small.

step_x = (xmax - xmin) / n_tasks
# ^ Width of each vertical strip in metres.

# ── Console summary ───────────────────────────────────────────────────────────
print(f'Total estimated size: {total_gb:.1f} GB')
print(f'Target size per task: {TARGET_GB} GB')
print(f'Tasks to submit:      {n_tasks}')
print(f'CRS:                  {Export_CRS}')
print(f'Export scale:         {EXPORT_SCALE}m')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

# ── Loop: one task per strip ──────────────────────────────────────────────────
for i in range(n_tasks):

    x0 = xmin + step_x * i
    x1 = xmin + step_x * (i + 1)
    # ^ Left and right edges of this strip in Export_CRS metres.

    strip_geom = (
        ee.Geometry.Rectangle([x0, ymin, x1, ymax], Export_CRS, False)
          .intersection(dissolved, ee.ErrorMargin(1))
    )
    # ^ Build the strip rectangle then intersect with the actual AOI polygon.
    #   This ensures pixels outside the ROW boundary are excluded from the
    #   export region entirely — keeps file sizes tighter than using the
    #   full rectangle as the region.
    #   ErrorMargin(1) = 1 metre tolerance for the intersection operation.

    strip_area_m2 = (x1 - x0) * (ymax - ymin)
    strip_est_gb  = (strip_area_m2 / (EXPORT_SCALE ** 2) * BYTES_PER_PX) / BYTES_PER_GB
    # ^ Per-strip size estimate using the rectangle area (conservative upper
    #   bound — actual file will be smaller after AOI intersection trims it).

    task = ee.batch.Export.image.toDrive(
        image          = ready_to_export.reproject(crs=Export_CRS, scale=EXPORT_SCALE),
        # ^ Final reproject ensures output pixels are exactly EXPORT_SCALE metres
        #   in Export_CRS — no ambiguity about GEE's internal resampling pyramid.

        description    = f'{FILE_PREFIX}_{i:03d}',
        # ^ Label shown in the GEE Tasks panel (earthengine.google.com).
        #   Zero-padded to 3 digits so tasks sort correctly up to 999 strips.

        folder         = FOLDER,
        # ^ Target Google Drive folder. GEE creates it if it doesn't exist.

        fileNamePrefix = f'{FILE_PREFIX}_{i:03d}',
        # ^ Filename (without .tif extension). GEE appends the extension.
        #   If a file with this name already exists in Drive, GEE appends a
        #   numeric suffix rather than overwriting — watch for duplicates on reruns.

        region         = strip_geom,
        # ^ Spatial extent of this strip, already intersected with the AOI.

        scale          = EXPORT_SCALE,
        # ^ Output pixel size in metres. Must match the reproject() call above.

        crs            = Export_CRS,
        # ^ Output CRS. GEE reprojects on-the-fly if source data differs.

        maxPixels      = int(1e13)
        # ^ Safety cap on total pixel count. Default is 1e8, far too low for
        #   1 m exports. 1e13 ≈ 10 trillion pixels — effectively unlimited.
    )

    task.start()
    # ^ Non-blocking — all tasks queue simultaneously and run in parallel.
    #   Each task may produce multiple files in Drive (GEE's internal tiling)
    #   but they will mosaic seamlessly in ArcGIS Pro or QGIS.

    pad = len(str(n_tasks))
    print(f'[{i+1:>{pad}}/{n_tasks}]  {FILE_PREFIX}_{i:03d}  |  ~{strip_est_gb:.1f} GB  |  x: {x0:.0f} → {x1:.0f}')

# ── Final summary ─────────────────────────────────────────────────────────────
print(f'\n✓ All {n_tasks} tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/tasks')
print('Or run:     ee.batch.Task.list()')

In [16]:
# NEW CHM




# =============================================================================
# 0.6 m CHM EXTRACTION (NAIP) — CLEAN AOI-BASED EXPORT
# =============================================================================

import ee
import geemap

ee.Initialize()
# ^ Initializes Earth Engine session so all ee.* objects can communicate with the GEE servers.
#   Must be called once per runtime before using any dataset or export.

# =============================================================================
# USER CONFIGURATION
# =============================================================================

ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ AOI polygon FeatureCollection stored in Earth Engine assets.
#   This defines the spatial extent for filtering, masking, and export.

Export_CRS = 'EPSG:5070'
# ^ Output coordinate reference system (Albers Equal Area Conic).
#   Chosen for CONUS-scale analysis because it preserves area relationships
#   and minimizes distortion across large east-west extents.

FOLDER = 'GEE_Exports_CHM'
# ^ Google Drive folder where exported GeoTIFF tiles will be written.

FILE_PREFIX = 'MN_I-90_CHM'
# ^ Base filename for all exported rasters.
#   Each partition will append an index suffix (_000, _001, etc.).

EXPORT_SCALE = 0.6
# ^ Spatial resolution of output raster in meters.
#   Matches native resolution of NAIP-derived canopy height model (~0.6 m).

NODATA_VALUE = -9999
# ^ Standard GIS NoData sentinel value.
#   Ensures compatibility when mosaicking in ArcGIS Pro / QGIS / Python raster tools.

CHM_YEAR = 2023
# ^ Temporal filter applied to CHM collection.
#   Ensures consistent dataset version across the study area.

CHM_SCALE_FACTOR = 100.0
# ^ CHM values are stored as integers scaled by 100 in dataset.
#   Division converts values back into meters above ground.

# =============================================================================
# LOAD INPUT DATA
# =============================================================================

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads AOI polygon(s) from Earth Engine asset registry.

dissolved = export_poly.union().geometry()
# ^ Dissolves all AOI features into a single geometry.
#   Used for spatial filtering and bounding box generation.

chm_collection = ee.ImageCollection(
    'projects/naip-chm/assets/conus-structure-model'
)
# ^ NAIP canopy height model dataset (CONUS-wide, high-resolution vegetation structure).

chm_mosaic = (
    chm_collection
    .filter(ee.Filter.eq('year', CHM_YEAR))
    # ^ Selects only CHM tiles matching requested year.

    .filterBounds(export_poly)
    # ^ Restricts dataset to tiles intersecting AOI.
    #   Reduces computation and avoids unnecessary global processing.

    .mosaic()
    # ^ Stitches overlapping tiles into a single seamless raster.
    #   GEE resolves overlaps using internal ordering rules.

    .divide(CHM_SCALE_FACTOR)
    # ^ Converts integer-scaled values to floating-point meters.

    .rename('canopy_height_m')
    # ^ Assigns explicit band name for GIS clarity and downstream analysis.
)

# =============================================================================
# PREPARE EXPORT IMAGE
# =============================================================================

chm_export = (
    chm_mosaic
    .clip(export_poly)
    # ^ Clips raster strictly to AOI boundary.
    #   Removes external pixels outside corridor.

    .reproject(crs=Export_CRS, scale=EXPORT_SCALE)
    # ^ Forces output grid alignment in target CRS and resolution.
    #   Prevents resampling inconsistencies during export.
)

ready_to_export = (
    chm_export
    .toFloat()
    # ^ Ensures 32-bit floating point output.
    #   Required for continuous vegetation height values.

    .unmask(NODATA_VALUE)
    # ^ Fills masked pixels with NoData sentinel value.
    #   Important for ArcGIS Pro mosaicking and raster math stability.
)

# =============================================================================
# INTERACTIVE MAP (QC / VISUAL VALIDATION)
# =============================================================================

Map = geemap.Map()
Map.centerObject(export_poly, 11)
# ^ Centers map view on AOI at moderate zoom level (regional scale).

Map.add_basemap('SATELLITE')
# ^ Adds high-resolution imagery for visual interpretation and validation.

Map.addLayer(
    chm_mosaic.clip(export_poly),
    {
        'min': 0,
        'max': 40,
        # ^ Visualization range based on expected vegetation height distribution.

        'palette': [
            'ffffff',  # bare ground / no canopy
            'c7e9b4',  # low vegetation
            '7fcdbb',  # shrub / small trees
            '41b6c4',  # medium canopy
            '2c7fb8',  # tall forest
            '253494'   # dense high canopy
        ]
    },
    'CHM (m)'
)
# ^ Adds CHM visualization layer for quality control.

Map.addLayer(export_poly, {'color': 'FF1744'}, 'AOI')
# ^ Overlays AOI boundary for spatial verification.

display(Map)
# ^ Renders interactive map in Jupyter/ArcGIS notebook environment.

# =============================================================================
# PARTITIONING LOGIC (AOI-BASED EXPORT SCALING)
# =============================================================================

projected_bbox = dissolved.bounds(1, Export_CRS)
# ^ Computes axis-aligned bounding box of AOI in projected CRS units (meters).

coords = projected_bbox.coordinates().get(0).getInfo()
# ^ Converts EE geometry coordinates to Python list for numerical processing.

xmin = coords[0][0]
ymin = coords[0][1]
xmax = coords[2][0]
ymax = coords[2][1]

bbox_area_m2 = projected_bbox.area(1).getInfo()
# ^ Computes total bounding box area in square meters.
#   Used for estimating export size and task splitting.

BYTES_PER_PX = 4
# ^ Float32 pixel size = 4 bytes per pixel.

BYTES_PER_GB = 1e9
# ^ Conversion factor for gigabytes (Earth Engine standard approximation).

total_gb = (bbox_area_m2 / (EXPORT_SCALE ** 2)) * BYTES_PER_PX / BYTES_PER_GB
# ^ Estimates total raster size in gigabytes:
#   area / pixel_area → number of pixels → bytes → GB

TARGET_GB = 10.0
# ^ Target maximum size per export task.
#   Prevents GEE export failures due to size limits.

n_tasks = max(1, int(total_gb / TARGET_GB) + 1)
# ^ Determines number of partitions required to stay under target size.

step_x = (xmax - xmin) / n_tasks
# ^ Width of each vertical partition in projected coordinate space.

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"Estimated size (GB): {total_gb:.2f}")
print(f"Number of tasks:     {n_tasks}")
print(f"CRS:                 {Export_CRS}")
print(f"Resolution:          {EXPORT_SCALE} m")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# =============================================================================
# EXPORT TASK EXECUTION (PARTITIONED AOI STRIPS)
# =============================================================================

for i in range(n_tasks):

    x0 = xmin + step_x * i
    x1 = xmin + step_x * (i + 1)
    # ^ Defines horizontal extent of each partition.

    strip_geom = (
        ee.Geometry.Rectangle([x0, ymin, x1, ymax], Export_CRS, False)
        .intersection(dissolved, ee.ErrorMargin(1))
    )
    # ^ Creates strip geometry and intersects with AOI.
    #   Ensures export only includes valid corridor pixels.

    task = ee.batch.Export.image.toDrive(
        image=ready_to_export,
        # ^ Final prepared raster for export (CHM, clipped, masked, formatted).

        description=f"{FILE_PREFIX}_{i:03d}",
        # ^ Task name displayed in Earth Engine Tasks panel.

        folder=FOLDER,
        # ^ Google Drive output folder.

        fileNamePrefix=f"{FILE_PREFIX}_{i:03d}",
        # ^ Output filename prefix for GeoTIFFs.

        region=strip_geom,
        # ^ Spatial extent of this export task.

        scale=EXPORT_SCALE,
        # ^ Output resolution in meters.

        crs=Export_CRS,
        # ^ Output coordinate reference system.

        maxPixels=int(1e13)
        # ^ Prevents export failure for large rasters.
    )

    task.start()
    # ^ Submits export task asynchronously to Earth Engine servers.

    print(f"[{i+1}/{n_tasks}] {FILE_PREFIX}_{i:03d}")

# =============================================================================
# FINAL STATUS OUTPUT
# =============================================================================

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("✓ CHM EXPORT TASKS SUBMITTED SUCCESSFULLY")
print("Monitor tasks at:")
print("https://code.earthengine.google.com/tasks")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

Map(center=[43.6353558671575, -95.7074188817978], controls=(WidgetControl(options=['position', 'transparent_bg…

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Estimated size (GB): 7.27
Tasks: 1
CRS: EPSG:5070
Scale: 0.6 m
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[1/1] MN_I-90_CHM_000
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✓ CHM EXPORT TASKS SUBMITTED
Monitor: https://code.earthengine.google.com/tasks
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:
# OLD

# 0.6m or 1m CHM Extraction 
# https://github.com/smorf-ntsg/naip-chm


# ═════════════════════════════════════════════════════════════════════════════
# 0.6 m NAIP CHM EXTRACTION — SHARDED EXPORT 
# ═════════════════════════════════════════════════════════════════════════════


# ═════════════════════════════════════════════════════════════════════════════
# 🎯 USER CONFIGURATION  —  only edit values in this block
# ═════════════════════════════════════════════════════════════════════════════

ASSET_ID    = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ Input AOI polygon(s). Must be a FeatureCollection in GEE.

Export_CRS  = 'EPSG:5070'
# ^ Albers Equal Area Conic (metres-based CRS suitable for CONUS-scale work).

FOLDER      = 'GEE_Exports_CHM'
# ^ Google Drive output folder. Auto-created if it does not exist.

FILE_PREFIX = 'naip_chm'
# ^ Output naming prefix → naip_chm_000.tif, naip_chm_001.tif, etc.

CHM_YEAR    = 2023
# ^ NAIP-CHM year filter.

# ── Export control ────────────────────────────────────────────────────────────
EXPORT_SCALE = 0.6
# ^ Output resolution in metres (native NAIP-CHM resolution ~0.6 m).

TARGET_GB    = 10.0
# ^ Maximum estimated size per export task.
#   Used to split AOI into multiple parallel export jobs.

# ═════════════════════════════════════════════════════════════════════════════
# CONSTANTS  —  no need to change these
# ═════════════════════════════════════════════════════════════════════════════

BYTES_PER_PX     = 4
# ^ Float32 = 4 bytes per pixel.

BYTES_PER_GB     = 1e9
# ^ Standard GB conversion used for estimates.

CHM_SCALE_FACTOR = 100.0
# ^ NAIP-CHM values are stored as integers ×100 → convert to metres.

# ═════════════════════════════════════════════════════════════════════════════
# LOAD INPUT DATA
# ═════════════════════════════════════════════════════════════════════════════

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads AOI polygons from your GEE asset.

dissolved = export_poly.union().geometry()
# ^ Merges all polygons into a single geometry used for tiling + filtering.

chm_collection = ee.ImageCollection(
    'projects/naip-chm/assets/conus-structure-model'
)
# ^ NAIP canopy height model (0.6 m resolution, CONUS-wide dataset).

chm_mosaic = (
    chm_collection
    .filter(ee.Filter.eq('year', CHM_YEAR))   # select target year
    .filterBounds(export_poly)                # keep only tiles intersecting AOI
    .mosaic()                                 # stitch into seamless raster
    .divide(CHM_SCALE_FACTOR)                 # convert integer-scaled values → metres
    .rename('canopy_height_m')               # assign meaningful band name
)
# ^ Final CHM raster in metres above ground.

# ═════════════════════════════════════════════════════════════════════════════
# PREPARE IMAGE FOR EXPORT
# ═════════════════════════════════════════════════════════════════════════════

row_mask = ee.Image.constant(1).clip(export_poly).unmask(0)
# ^ Binary mask:
#   1 = inside AOI
#   0 = outside AOI

ready_to_export = (
    chm_mosaic
    .updateMask(row_mask)   # mask everything outside AOI
    .unmask(-1)             # fill masked pixels with -1 (NoData sentinel)
    .toFloat()              # enforce Float32 output (matches BYTES_PER_PX)
)
# ^ Final export image:
#   valid CHM values → 0–40 m
#   outside AOI      → -1
#   no data         → -1

# ═════════════════════════════════════════════════════════════════════════════
# INTERACTIVE MAP PREVIEW
# ═════════════════════════════════════════════════════════════════════════════

chm_vis_params = {
    'min': 0,
    'max': 40,
    'palette': [
        'ffffff',  # bare ground
        'c7e9b4',
        '7fcdbb',
        '41b6c4',
        '2c7fb8',
        '253494'   # tall canopy
    ]
}

Map = geemap.Map()
Map.centerObject(export_poly, 10)     # zoom to corridor / county scale
Map.add_basemap('SATELLITE')          # high-resolution imagery background

Map.addLayer(
    chm_mosaic.clip(export_poly),
    chm_vis_params,
    'CHM 2023'
)
# ^ Visual preview of canopy height model.

Map.addLayer(
    export_poly,
    {'color': 'FF0000', 'fillColor': '00000000'},
    'AOI'
)
# ^ Red outline of analysis region.

display(Map)

# ═════════════════════════════════════════════════════════════════════════════
# AUTO-SHARDED EXPORT — SPLITS AOI INTO MULTIPLE TASKS
# ═════════════════════════════════════════════════════════════════════════════

projected_bbox = dissolved.bounds(1, Export_CRS)
# ^ Compute bounding box of AOI in projected coordinate system.

coords = projected_bbox.coordinates().get(0).getInfo()
# ^ Extract bounding box coordinates into Python.

xmin = coords[0][0]
ymin = coords[0][1]
xmax = coords[2][0]
ymax = coords[2][1]

bbox_area_m2 = projected_bbox.area(1).getInfo()
# ^ Total AOI bounding box area in square metres.

total_gb = (bbox_area_m2 / (EXPORT_SCALE ** 2) * BYTES_PER_PX) / BYTES_PER_GB
# ^ Estimate total export size in GB.

n_tasks = max(1, int(total_gb / TARGET_GB) + 1)
# ^ Number of export tasks needed so each stays under TARGET_GB.

step_x = (xmax - xmin) / n_tasks
# ^ Width of each vertical strip in projected metres.

# ── Console summary ───────────────────────────────────────────────────────────
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Total estimated size (GB): {total_gb:.2f}')
print(f'Target size per task:      {TARGET_GB} GB')
print(f'Export CRS:                {Export_CRS}')
print(f'Export scale:              {EXPORT_SCALE} m')
print(f'Tasks to submit:           {n_tasks}')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

# ═════════════════════════════════════════════════════════════════════════════
# LOOP THROUGH STRIPS — CREATE ONE EXPORT TASK PER STRIP
# ═════════════════════════════════════════════════════════════════════════════

for i in range(n_tasks):

    x0 = xmin + step_x * i
    x1 = xmin + step_x * (i + 1)
    # ^ Define horizontal bounds of this strip.

    strip_geom = (
        ee.Geometry.Rectangle([x0, ymin, x1, ymax], Export_CRS, False)
          .intersection(dissolved, ee.ErrorMargin(1))
    )
    # ^ Clip strip rectangle to actual AOI boundary.
    #   Prevents exporting unnecessary pixels outside corridor.

    strip_est_gb = (
        ((x1 - x0) * (ymax - ymin))
        / (EXPORT_SCALE ** 2)
        * BYTES_PER_PX
        / BYTES_PER_GB
    )
    # ^ Conservative per-strip size estimate (pre-intersection).

    task = ee.batch.Export.image.toDrive(
        image = ready_to_export.reproject(crs=Export_CRS, scale=EXPORT_SCALE),
        # ^ Force consistent pixel grid in output CRS.

        description = f'{FILE_PREFIX}_{i:03d}',
        # ^ Task name in GEE Tasks panel.

        folder = FOLDER,
        # ^ Google Drive destination folder.

        fileNamePrefix = f'{FILE_PREFIX}_{i:03d}',
        # ^ Output filename prefix.

        region = strip_geom,
        # ^ Spatial extent of this export task.

        scale = EXPORT_SCALE,
        # ^ Output pixel resolution.

        crs = Export_CRS,
        # ^ Output coordinate reference system.

        maxPixels = int(1e13)
        # ^ Prevents GEE export limit errors for large datasets.
    )

    task.start()
    # ^ Submits task asynchronously to Earth Engine queue.

    print(
        f'[{i+1}/{n_tasks}] '
        f'{FILE_PREFIX}_{i:03d}  |  ~{strip_est_gb:.2f} GB'
    )

# ── Final status ──────────────────────────────────────────────────────────────
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'✓ All {n_tasks} CHM export tasks submitted')
print('Monitor at: https://code.earthengine.google.com/tasks')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

In [ ]:
# =============================================================================
# MOSAIC RASTERS — ArcGIS Pro (NoData = -9999)
# =============================================================================
# Run in ArcGIS Pro Python Notebook

import arcpy
import os

# ----------------------------------------
# 1️⃣ INPUTS
# ----------------------------------------

shard_dir  = r"C:\Users\KyleSteen.AzureAD\Downloads\slope_Mosaic"
output_gdb = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\Jessie_Atlanta_CHM\WashDOT.gdb"
output_name = "Mosaic_1"

NODATA_VALUE = -9999

# ----------------------------------------
# 2️⃣ COLLECT RASTERS
# ----------------------------------------

shards = [
    os.path.join(shard_dir, f)
    for f in os.listdir(shard_dir)
    if f.lower().endswith(".tif")
]

if not shards:
    raise FileNotFoundError(f"No .tif files found in: {shard_dir}")

shards = sorted(shards)

print(f"Found {len(shards)} raster(s)")

# ----------------------------------------
# 3️⃣ FORCE NODATA STANDARDIZATION
# ----------------------------------------
# This step actually WRITES nodata into the raster

print("Standardizing NoData = -9999...")

clean_shards = []

for shard in shards:
    out_raster = shard.replace(".tif", "_nd.tif")

    arcpy.management.CopyRaster(
        in_raster=shard,
        out_rasterdataset=out_raster,
        nodata_value=str(NODATA_VALUE),
        pixel_type="32_BIT_FLOAT"
    )

    clean_shards.append(out_raster)

print("NoData standardization complete.")

# ----------------------------------------
# 4️⃣ MOSAIC TO NEW RASTER
# ----------------------------------------

input_rasters = ";".join(clean_shards)

print("Mosaicking rasters...")

with arcpy.EnvManager(parallelProcessingFactor="80%"):
    arcpy.management.MosaicToNewRaster(
        input_rasters=input_rasters,
        output_location=output_gdb,
        raster_dataset_name_with_extension=output_name,
        coordinate_system_for_the_raster=arcpy.SpatialReference(26915),
        pixel_type="32_BIT_FLOAT",
        number_of_bands=1,
        mosaic_method="FIRST",
        mosaic_colormap_mode="FIRST"
    )

print(f"\nComplete: {os.path.join(output_gdb, output_name)}")

In [ ]:
# OLD

# Use this Script to Mosaic the Imagery. Since it requires ArcPy, it is best to run in ArcGIS Pro Python Notebook
# Note: This can take a very long time. For the MN I-90 study area, it took over 1 hour

import arcpy
import os

# ----------------------------------------
# 1️⃣ Point to your shard directory and output
# ----------------------------------------
shard_dir  = r"C:\Users\KyleSteen.AzureAD\Downloads\slope_Mosaic" # Folder with the individual .tif files
output_gdb = r"C:\Users\KyleSteen.AzureAD\Documents\ArcGIS\Projects\Jessie_Atlanta_CHM\WashDOT.gdb" # Output .gdb
output_name = "Mosaic_1" # Name of output Mosaic

# ----------------------------------------
# 2️⃣ Collect all .tif files in directory
# ----------------------------------------
shards = [
    os.path.join(shard_dir, f)
    for f in os.listdir(shard_dir)
    if f.lower().endswith(".tif")
]

if not shards:
    raise FileNotFoundError(f"No .tif files found in: {shard_dir}")

print(f"Found {len(shards)} shard(s):")
for s in sorted(shards):
    print(f"  {os.path.basename(s)}")

# ----------------------------------------
# 3️⃣ Tag nodata = -1 on every shard
# ----------------------------------------
print("\nSetting nodata = -1 on all shards...")
for shard in shards:
    arcpy.management.SetRasterProperties(shard, nodata="1 -1")
print("Done.")

# ----------------------------------------
# 4️⃣ Mosaic to new raster
# ----------------------------------------
input_rasters = ";".join(sorted(shards))

print(f"\nMosaicking to {output_name}...")
with arcpy.EnvManager(parallelProcessingFactor="80%"):
    arcpy.management.MosaicToNewRaster(
        input_rasters=input_rasters,
        output_location=output_gdb,
        raster_dataset_name_with_extension=output_name,
        coordinate_system_for_the_raster=arcpy.SpatialReference(26915), # Ensure Correct Coordinate System
        pixel_type="32_BIT_FLOAT",
        cellsize=None,
        number_of_bands=1,
        mosaic_method="FIRST",
        mosaic_colormap_mode="FIRST"
    )

print(f"\nComplete. Output: {os.path.join(output_gdb, output_name)}")